# Module 46 — Churn Prevention Campaign: High-Value User Targeting

> **Track 4 · Marketing Analytics & Optimization**  
> **Difficulty**: Advanced  
> **Runtime**: ~10 minutes  
> **Key Libraries**: Scikit-learn, Pandas, NumPy, Plotly  
> **Prerequisite**: Module 18 (Churn Prediction), Module 40 (CLV)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Customer Data](#3-synthetic-customer-data)
4. [Churn Model](#4-churn-model)
5. [CLV Calculation](#5-clv-calculation)
6. [Campaign Targeting](#6-campaign-targeting)
7. [Visualization](#7-visualization)
8. [Key Business Takeaway](#8-key-business-takeaway)

---
## §1 · Business Context

### Why combine churn prediction with CLV?

Churn prevention campaigns should **prioritize high-value customers**:
- **Cost efficiency**: Retention is cheaper than acquisition.
- **ROI maximization**: Focus on customers with highest CLV.
- **Resource allocation**: Limited retention budget.

**Applications**:
1. **Retention campaigns**: Target high-CLV at-risk customers.
2. **Win-back campaigns**: Re-engage churned high-value customers.
3. **Loyalty programs**: Reward high-CLV customers.

**Key metrics**:
- **Expected loss**: CLV × Churn probability.
- **Retention ROI**: (Saved CLV - Campaign cost) / Campaign cost.

In this notebook we will:
1. Generate synthetic customer data with churn and CLV.
2. Build churn prediction model.
3. Calculate expected loss for each customer.
4. Design targeted retention campaign.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Scikit-learn ─────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")

---
## §3 · Synthetic Customer Data

In [ ]:
# Generate synthetic customer data
N_CUSTOMERS = 10000

data = {
    'customer_id': range(N_CUSTOMERS),
    'age': np.random.normal(40, 12, N_CUSTOMERS).clip(18, 80).astype(int),
    'tenure_months': np.random.exponential(24, N_CUSTOMERS).astype(int),
    'monthly_spend': np.random.exponential(100, N_CUSTOMERS),
    'purchase_frequency': np.random.poisson(5, N_CUSTOMERS),
    'support_tickets': np.random.poisson(2, N_CUSTOMERS),
    'days_since_last_login': np.random.exponential(15, N_CUSTOMERS).astype(int),
}

df = pd.DataFrame(data)

# Generate churn labels
churn_prob = (
    0.1
    + 0.2 * (df['days_since_last_login'] > 30).astype(float)
    + 0.15 * (df['support_tickets'] > 3).astype(float)
    - 0.1 * (df['tenure_months'] > 12).astype(float)
    + np.random.normal(0, 0.05, N_CUSTOMERS)
)
churn_prob = churn_prob.clip(0.01, 0.99)
df['churned'] = np.random.binomial(1, churn_prob)

# Calculate CLV (simplified)
df['clv'] = df['monthly_spend'] * df['tenure_months'] * 0.3  # 30% margin

print(f"✓ Generated {N_CUSTOMERS:,} customers")
print(f"  Churn rate: {df['churned'].mean()*100:.1f}%")
print(f"  Average CLV: ${df['clv'].mean():,.2f}")
print(f"  Total CLV: ${df['clv'].sum():,.2f}")

---
## §4 · Churn Model

In [ ]:
# Build churn prediction model
feature_cols = ['age', 'tenure_months', 'monthly_spend', 'purchase_frequency',
                'support_tickets', 'days_since_last_login']

X = df[feature_cols]
y = df['churned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train model
model = GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)

# Predict churn probability
df['churn_probability'] = model.predict_proba(X)[:, 1]

auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

print(f"✓ Churn model trained")
print(f"  AUC: {auc:.4f}")
print(f"  Average churn probability: {df['churn_probability'].mean()*100:.1f}%")

---
## §5 · CLV Calculation

In [ ]:
# Calculate expected loss from churn
df['expected_loss'] = df['churn_probability'] * df['clv']

# Rank customers by expected loss
df_ranked = df.sort_values('expected_loss', ascending=False)

print(f"✓ Expected loss calculated")
print(f"  Total expected loss: ${df['expected_loss'].sum():,.2f}")
print(f"  Average expected loss: ${df['expected_loss'].mean():,.2f}")
print(f"\nTop 10 customers by expected loss:")
print(df_ranked.head(10)[['customer_id', 'clv', 'churn_probability', 'expected_loss']])

---
## §6 · Campaign Targeting

In [ ]:
# Design retention campaign
CAMPAIGN_COST_PER_CUSTOMER = 50  # $50 per targeted customer
RETENTION_RATE = 0.30  # 30% of targeted customers stay

# Target top 20% by expected loss
target_pct = 0.20
n_target = int(N_CUSTOMERS * target_pct)
df['targeted'] = 0
df.loc[df_ranked.head(n_target).index, 'targeted'] = 1

# Calculate campaign ROI
targeted_customers = df[df['targeted'] == 1]
campaign_cost = n_target * CAMPAIGN_COST_PER_CUSTOMER
saved_clv = targeted_customers['expected_loss'].sum() * RETENTION_RATE
roi = (saved_clv - campaign_cost) / campaign_cost * 100

print(f"✓ Campaign targeting complete")
print(f"  Targeted customers: {n_target:,} ({target_pct*100:.0f}%)")
print(f"  Campaign cost: ${campaign_cost:,.2f}")
print(f"  Expected saved CLV: ${saved_clv:,.2f}")
print(f"  ROI: {roi:.1f}%")
print(f"\nTargeted customer characteristics:")
print(f"  Average CLV: ${targeted_customers['clv'].mean():,.2f}")
print(f"  Average churn prob: {targeted_customers['churn_probability'].mean()*100:.1f}%")

---
## §7 · Visualization

In [ ]:
# Visualize campaign targeting
fig = make_subplots(rows=1, cols=2, subplot_titles=['Expected Loss Distribution', 'Targeted vs Not Targeted'])

fig.add_trace(go.Histogram(
    x=df['expected_loss'],
    nbinsx=50,
    name='Expected Loss'
), row=1, col=1)

fig.add_trace(go.Box(
    y=df[df['targeted']==1]['clv'],
    name='Targeted',
    marker_color='#EF553B'
), row=1, col=2)

fig.add_trace(go.Box(
    y=df[df['targeted']==0]['clv'],
    name='Not Targeted',
    marker_color='#636EFA'
), row=1, col=2)

fig.update_layout(height=400, showlegend=True)
fig.show()

print("💡 Key insights:")
print("   - Targeting top 20% by expected loss maximizes ROI")
print("   - High-CLV customers with high churn risk are top priorities")
print("   - Campaign ROI is positive with 30% retention rate")

---
## §8 · Key Business Takeaway

> ### 🎯 Action Items for Retention Teams
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | Retention campaigns, win-back, loyalty programs |
> | **Targeting** | High expected loss (CLV × Churn probability) |
> | **Budget** | Allocate based on expected loss ranking |
> | **Metrics** | Retention rate, campaign ROI, saved CLV |
> | **Frequency** | Monthly targeting, quarterly campaign review |
>
> ### 💡 Campaign Design
>
> | Customer Segment | Strategy | Expected Impact |
> |------------------|----------|------------------|
> | High CLV + High Churn | Personal outreach, premium offers | 30-40% retention |
> | High CLV + Low Churn | Loyalty rewards, upsell | 5-10% increase |
> | Low CLV + High Churn | Automated campaigns | 10-20% retention |
> | Low CLV + Low Churn | No action | N/A |
>
> ### 🔑 Key Takeaways
>
> 1. **Expected loss = CLV × Churn probability** guides targeting.
> 2. **Focus retention budget** on high expected loss customers.
> 3. **Campaign ROI** can be calculated before running.
> 4. **Combining churn and CLV** maximizes retention ROI.
> 5. **Regular targeting refresh** captures changing customer behavior.